# Lifecycle: Startup, running and shutdown

**Every MCP connection goes through three phases: Initialization (the handshake), operation (listing and calling tools), and shutdown. A server can also run its own code at startup and shutdown, called a lifespan, to open a resource like a database once, instead of on every tool call.**

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 client side                    server side
 -----------                    -----------
 stdio_client starts it         lifespan startup: Open the database
 initialize()        phase 1    replies to the handshake
 call_tool(...)      phase 2    tools use the open database
 leaving async with  phase 3    lifespan shutdown: Close it
</pre>

Read `../2-server-stdio-http-sse/client-stdio.ipynb` first.

## 1. The client side: The three phases

You've already used all three:

- **Initialization:** `session.initialize()`, the handshake from section 6 of `../1-understanding-mcp/README.md`. The client and server agree on a protocol version and say what they support. The newest spec (2026-07-28) drops this step, as the README's appendix explains.
- **Operation:** `list_tools()` and `call_tool(...)`, as many times as you like.
- **Shutdown:** Leaving the `async with` blocks. `ClientSession` closes, then `stdio_client` closes the server's stdin, and the server process ends.

## 2. The server side: A lifespan

<div style="font-size: 0.85em">

```python
import sqlite3
import sys
from contextlib import asynccontextmanager

from mcp.server.fastmcp import Context, FastMCP


@asynccontextmanager
async def lifespan(server):
    print("Startup: Opening the database", file=sys.stderr)
    db = sqlite3.connect(":memory:")
    db.execute("CREATE TABLE notes (text TEXT)")
    db.execute("INSERT INTO notes VALUES ('Standup at 10'), ('Demo on Friday')")

    try:
        yield {"db": db}
    finally:
        db.close()
        print("Shutdown: Closed the database", file=sys.stderr)


mcp = FastMCP("Notes", lifespan=lifespan)


@mcp.tool()
def count_notes(ctx: Context) -> int:
    """Count the notes in the database"""
    db = ctx.request_context.lifespan_context["db"]
    row = db.execute("SELECT COUNT(*) FROM notes").fetchone()
    return row[0]


@mcp.tool()
def add_note(text: str, ctx: Context) -> str:
    """Add a note to the database"""
    db = ctx.request_context.lifespan_context["db"]
    db.execute("INSERT INTO notes VALUES (?)", (text,))
    return "Added: " + text


if __name__ == "__main__":
    mcp.run(transport="stdio")
```

</div>

- **SQLite:** The database built into Python (`sqlite3`). `":memory:"` keeps it in memory, so it disappears when the server stops.
- **`@asynccontextmanager`:** From Python's `contextlib`. It turns a function with one `yield` into something `async with` can use. The code before `yield` runs on entry, and the code after it runs on exit. For a lifespan, that means before `yield` is startup, and after it is shutdown.
- **`yield {"db": db}`:** The dict becomes the server's `lifespan_context`, which every tool can reach.
- **`try` and `finally`:** The `finally` part runs even if the server crashes, so the database always closes.
- **`FastMCP("Notes", lifespan=lifespan)`:** Tells FastMCP to wrap the server's whole run in the lifespan.
- **`ctx: Context`:** FastMCP fills a parameter of type `Context` on every call. It holds the request's details, including `ctx.request_context.lifespan_context`.

The `ctx` parameter isn't part of either tool's input schema, so the model never sees it:

In [1]:
import server

tools = await server.mcp.list_tools()
for tool in tools:
    print(tool.name)
    print(tool.inputSchema)

count_notes
{'properties': {}, 'title': 'count_notesArguments', 'type': 'object'}
add_note
{'properties': {'text': {'title': 'Text', 'type': 'string'}}, 'required': ['text'], 'title': 'add_noteArguments', 'type': 'object'}


## 3. Watching the lifecycle

The client counts the notes, adds one, and counts again, all on one connection:

In [2]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

notes_server = StdioServerParameters(command="python", args=["server.py"])

with open("server_log.txt", "w") as log:
    async with stdio_client(notes_server, errlog=log) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            result = await session.call_tool("count_notes", {})
            print(result.content[0].text)

            result = await session.call_tool("add_note", {"text": "Book the meeting room"})
            print(result.content[0].text)

            result = await session.call_tool("count_notes", {})
            print(result.content[0].text)

2
Added: Book the meeting room
3


The count went from 2 to 3, because every call used the same open database. The server's log shows the whole lifecycle:

In [3]:
with open("server_log.txt") as log:
    print(log.read())

Startup: Opening the database
Processing request of type CallToolRequest
Processing request of type ListToolsRequest
Processing request of type CallToolRequest
Processing request of type CallToolRequest
Shutdown: Closed the database



- **"Startup" once at the top:** The lifespan opened the database when the server started.
- **"Shutdown" once at the bottom:** It closed the database when the client left the `async with` blocks.
- **The one `tools/list`:** The client SDK sent it by itself after the first call, to check results against each tool's output schema.

## 4. Why a lifespan

Without one, each tool would have to open and close the database on every call, which is slow, or keep it in a global variable that never gets closed. A lifespan opens it once, shares it with every tool, and closes it cleanly. That's the same pattern for any expensive resource: A database pool, an HTTP client, or a loaded ML model.

## 5. The .py version

`client.py` does the same, then prints the log. Run from this folder with `python client.py`:

<div style="font-size: 0.85em">

```text
2
Added: Book the meeting room
3
Startup: Opening the database
Processing request of type CallToolRequest
Processing request of type ListToolsRequest
Processing request of type CallToolRequest
Processing request of type CallToolRequest
Shutdown: Closed the database
```

</div>

## Say it out loud

> "An MCP connection has three phases: Initialization, where the client and server shake hands and agree on a protocol version, then operation, with the tool calls, then shutdown, when the client closes the transport. On the server, FastMCP takes a lifespan, an async context manager. The code before `yield` runs at startup, the code after it runs at shutdown, and what it yields is shared with every tool through `ctx.request_context.lifespan_context`. I used it to open a database once and close it cleanly, instead of opening it on every call."